# Polymerisation kinetics and molecular-weight distributions

*chemistry · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/chemistry/polymerisation_mwd)

Why are step-growth polymers (polyesters, nylons) so polydisperse, while living polymerisation makes nearly uniform chains? A population balance over chain lengths, one $(S, L)$ tensor for both mechanisms, reproduces Carothers' $X_n = 1/(1-p)$ and Flory's most probable distribution $w(x) = x(1-p)^2p^{x-1}$ with dispersity $Đ \to 2$, checked against a kinetic Monte Carlo of 100 000 monomers, and the Poisson distribution of living chains with $Đ \approx 1 + 1/X_n$.

**Vector form**

$$\frac{dP_x}{d\tau} = \sum_{y=1}^{x-1}P_yP_{x-y} - 2P_xN,\qquad \frac{dP^*_x}{d\nu} = P^*_{x-1} - P^*_x,\qquad X_n = \frac{1}{1-p},\quad w(x) = x(1-p)^2p^{x-1},\quad Đ = \frac{X_w}{X_n}$$

**Index (tensor) form**

$$\dot P_{0x} = T_{xy}P_{0y} - 2P_{0x}P_{0y}\mathbb 1_y,\quad T_{xy} = P_{0,x-y},\qquad \dot P_{1x} = P_{1,x-1} - P_{1x},\qquad \mu_{sk} = P_{sx}\,x^k,\qquad Đ_s = \frac{\mu_{s2}\,\mu_{s0}}{\mu_{s1}^2}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $P_{sx}$ | `traj[k]` | (2, L) | chains of length $x$ ($x = 1 … L$) for mechanism $s$: 0 = step growth (per monomer), 1 = living (per initiator) |
| $s = X_n - 1$ | `s` | (T,) | one clock for both rows: $\tau = kN_0t$ for step growth, $\nu = \int k_p[M]\,dt$ for living growth |
| $T_{xy} = P_{0,x-y}$ | `P[idx] * mask` | (L, L) | Toeplitz matrix gathered from the populations; zero where $x - y < 1$ |
| $T_{xy}P_{0y}$ | `einsum('xy,y->x', T, P)` | (L,) | repeated $y$ ⇒ summed: every way to join a $y$-mer to an $(x-y)$-mer |
| $2P_{0x}N$ | `2.0 * P * P.sum()` | (L,) | a chain reacts through either of its two ends with any of the $N$ chains |
| $P_{1,x-1} - P_{1x}$ | `living_rhs(P[1])` | (L,) | living chains only add monomer, one unit at a time, and never terminate |
| $\mu_{sk} = P_{sx}x^k$ | `einsum('...x,kx->...k', P, powers)` | (..., 3) | $x$ summed: moments $k = 0, 1, 2$ give $X_n = \mu_1/\mu_0$, $X_w = \mu_2/\mu_1$ |
| $(1-p)^2p^{x-1}$ | `flory_number(x, p) * (1 - p)` | (K, L) | exact solution for step growth: chains per initial monomer at conversion $p = \tau/(1+\tau)$ |
| $e^{-\nu}\nu^{x-1}/(x-1)!$ | `poisson(x, nu)` | (K, L) | exact solution for living growth, with lgamma for the factorial |
| $P^{\mathrm{MC}}_x$ | `kinetic_monte_carlo(n0, events, L, seed)` | (K, L) | join two random chains one bond at a time: an independent stochastic check |

Step growth: every molecule carries reactive ends (here one A and one B, like a hydroxy acid) and ANY two chains can join, monomer with monomer, dimer with 50-mer. Flory's equal-reactivity assumption (the rate does not depend on chain length) turns the population balance into the Smoluchowski coagulation equation with a constant kernel, whose exact solution is the most probable distribution: a fraction $p$ of the ends has reacted, so a chain of length $x$ is a run of $x - 1$ reacted bonds ended by an unreacted one, with probability $(1-p)p^{x-1}$. Its averages are Carothers' $X_n = 1/(1-p)$ and $X_w = (1+p)/(1-p)$, so $Đ = 1 + p \to 2$. The practical lesson is harsh: 98 % conversion gives only $X_n = 50$, and long chains need $p > 0.99$ and a perfect 1:1 balance of reactive groups. Living chain growth (anionic, or controlled radical methods such as ATRP or RAFT, which approximate it) is the opposite: all chains start together and grow one monomer at a time without termination, so after $\nu$ additions per chain the lengths are Poisson distributed and $Đ = 1 + \nu/(1+\nu)^2 \approx 1 + 1/X_n$ falls towards 1. The tensor form keeps both mechanisms in one $(S, L)$ tensor advanced by RK4. The step-growth convolution $\sum_y P_y P_{x-y}$ is an einsum with a Toeplitz matrix gathered from $P$ (an FFT would do the same in $O(L\log L)$). Because the two rows use clocks with $X_n = 1 + s$, each frame compares the two mechanisms at the same $X_n$, on an axis $x / X_n$ where the Flory shape is fixed and the Poisson peak narrows. The kinetic Monte Carlo joins random pairs of chains; it agrees with the population balance up to sampling noise, which grows near the end because only $N_0(1-p)$ chains are left (2000 at $p = 0.98$). Simplifications: no ring formation, no side reactions, a perfectly balanced A–B stoichiometry, chain lengths cut off at $L$ (the omitted tail is below $10^{-6}$), and instantaneous initiation with no termination or transfer for the living case. Conventional free-radical polymerisation, with termination, sits in between ($Đ \approx 1.5$–2).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/chemistry/polymerisation_mwd/main.py
"""Main loop: monomers → population balance (step + living growth) → kinetic Monte Carlo → compare → plot."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors

SEED = 0


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    p_max = params["p_max"]
    n0 = params["n_monomers"] // (10 if fast else 1)
    L = tensors.chain_length_cap(p_max)
    x = tensors.lengths(L, dev)  # (L,)
    idx, mask = tensors.shift_index(L, dev)  # (L, L)

    # --- population balance: both mechanisms in one (2, L) tensor, clock s = X_n - 1 ---
    t0 = time.perf_counter()
    s_max = p_max / (1 - p_max)  # Carothers: X_n - 1 = p / (1 - p)
    s, traj = solve.population_balance(tensors.initial_state(L, dev), idx, mask, s_max, 0.25 if fast else 0.1)
    t_ode = time.perf_counter() - t0

    # frames evenly spaced in conversion p (not in s, which races at the end)
    p_frames = torch.linspace(0, p_max, 90, dtype=torch.float64, device=dev)
    picks = torch.searchsorted(s, (p_frames / (1 - p_frames)).clamp_max(s[-1])).clamp_max(len(s) - 1).unique()
    s_k, P_k = s[picks], traj[picks]  # (K,), (K, 2, L)
    p_k = s_k / (1 + s_k)  # conversion of the step-growth row at each frame

    # --- kinetic Monte Carlo of step growth at the same conversions ---
    t0 = time.perf_counter()
    events = [int(round(p.item() * n0)) for p in p_k]
    kmc = solve.kinetic_monte_carlo(n0, events, L, SEED).to(dev)  # (K, L) chains per initial monomer
    t_kmc = time.perf_counter() - t0
    p_kmc = torch.tensor(events, dtype=torch.float64, device=dev) / n0

    xn, _, disp = equation.averages(P_k, x)  # (K, 2) each
    xn_mc, _, disp_mc = equation.averages(kmc, x)  # (K,)
    flory = equation.flory_number(x, p_k[:, None])  # (K, L) chains per initial monomer is (1-p) n(x)
    flory_err = (P_k[:, 0] - (1 - p_k[:, None]) * flory).abs().max().item()
    poisson = equation.poisson(x, s_k[:, None])  # (K, L)
    poisson_err = (P_k[:, 1] - poisson).abs().max().item()
    m_end = equation.moments(traj[-1], x)  # (2, 3)

    print(f"A–B step growth and living chain growth, chain lengths x = 1 … {L}, "
          f"RK4 over {len(s) - 1} steps in {t_ode:.1f} s; KMC with {n0:,} monomers in {t_kmc:.1f} s")  # fmt: skip
    print("   p      X_n: ODE  Carothers   KMC     Đ step: ODE   1 + p    KMC     Đ living: ODE  1+ν/(1+ν)²  1+1/X_n")
    shown = sorted({int(torch.argmin((p_k - q).abs())) for q in (0.5, 0.8, 0.9, 0.95, 0.98, 0.99, p_max) if q <= p_max})
    for k in shown:
        p, nu = p_k[k].item(), s_k[k].item()
        step = f"{xn[k, 0].item():8.2f} {equation.carothers(p):9.2f} {xn_mc[k].item():8.2f}"
        disp_step = f"{disp[k, 0].item():10.3f} {1 + p:8.3f} {disp_mc[k].item():7.3f}"
        disp_living = f"{disp[k, 1].item():12.4f} {1 + nu / (1 + nu) ** 2:10.4f} {1 + 1 / xn[k, 1].item():9.4f}"
        print(f"  {p:.3f}   {step}   {disp_step}   {disp_living}")
    print(f"population balance vs exact solutions: max |P - N₀(1-p)²p^(x-1)| = {flory_err:.1e},   "
          f"max |P - Poisson| = {poisson_err:.1e}")  # fmt: skip
    print(f"conserved: monomer units Σ x P_x = {m_end[0, 1].item():.6f} (step growth, per monomer), "
          f"living chains Σ P_x = {m_end[1, 0].item():.6f} (per initiator)")  # fmt: skip

    return plot.animate(
        x, p_k, s_k, P_k, poisson, kmc, p_kmc, xn, disp, xn_mc, disp_mc, out_dir / "polymerisation_mwd.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/chemistry/polymerisation_mwd/equation.py
r"""Physics → tensor translation.

STEP GROWTH (A–B monomers, e.g. a hydroxy acid making a polyester): any chain end A can react with any end B.
With Flory's equal-reactivity assumption every chain has one A and one B end and the same rate constant k, so
the population balance for P_x (chains of length x) is the Smoluchowski equation with a constant kernel:

    dP_x/dτ = Σ_y P_y P_{x-y} - 2 P_x N,       N = Σ_y P_y,       τ = k N_0 t

(y summed; the first term forms x from y + (x - y), the second loses x through either of its two ends).
Its exact solution is Flory's most probable distribution at conversion p = 1 - N/N_0 = τ/(1 + τ):

    n(x) = (1 - p) p^{x-1}        (number fraction),       w(x) = x (1 - p)² p^{x-1}    (weight fraction)

    X_n = 1/(1 - p)   (Carothers),     X_w = (1 + p)/(1 - p),     Đ = X_w / X_n = 1 + p  →  2

LIVING CHAIN GROWTH (anionic, ATRP, …): every initiator starts one chain at once, chains only add monomer,
nothing terminates. Measured in the kinetic chain length ν = ∫ k_p [M] dt (monomers added per chain):

    dP_x/dν = P_{x-1} - P_x       ⇒      P_x = e^{-ν} ν^{x-1} / (x-1)!    (Poisson)

    X_n = 1 + ν,       Đ = 1 + ν / (1 + ν)²  ≈  1 + 1/X_n

Both mechanisms are integrated together in one (S, L) tensor (S = 2 mechanisms, L chain lengths), with the
clock s = X_n - 1 (s = τ for step growth, s = ν for living growth), so every frame compares equal X_n.
"""

import torch

# --- step growth ---


def convolution(P: torch.Tensor, idx: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    r"""C_x = Σ_y P_y P_{x-y}   (y summed):  T_{xy} = P_{x-y} by gathering, then einsum('xy,y->x').   (L,) → (L,)."""
    T = P[idx] * mask  # (L, L) Toeplitz matrix of partner populations
    return torch.einsum("xy,y->x", T, P)


def step_growth_rhs(P: torch.Tensor, idx: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    r"""dP_x/dτ = Σ_y P_y P_{x-y} - 2 P_x N.   (L,) → (L,)."""
    return convolution(P, idx, mask) - 2.0 * P * P.sum()


def flory_number(x: torch.Tensor, p) -> torch.Tensor:
    """n(x) = (1 - p) p^{x-1}: the fraction of chains that have length x.   x (L,), p scalar or (F, 1)."""
    return (1 - p) * p ** (x - 1)


def flory_weight(x: torch.Tensor, p) -> torch.Tensor:
    """w(x) = x (1 - p)² p^{x-1}: the fraction of monomer units that sit in chains of length x."""
    return x * (1 - p) ** 2 * p ** (x - 1)


def carothers(p):
    """X_n = 1 / (1 - p): number-average degree of polymerisation at conversion p."""
    return 1.0 / (1.0 - p)


# --- living chain growth ---


def living_rhs(P: torch.Tensor) -> torch.Tensor:
    r"""dP_x/dν = P_{x-1} - P_x  (a chain of length x-1 gains a monomer, one of length x leaves).   (L,) → (L,)."""
    shifted = torch.cat([torch.zeros_like(P[:1]), P[:-1]])  # P_{x-1}, with P_0 = 0
    return shifted - P


def poisson(x: torch.Tensor, nu) -> torch.Tensor:
    r"""P_x = e^{-ν} ν^{x-1} / (x-1)!, written with lgamma so it stays finite for long chains."""
    nu = torch.as_tensor(nu, dtype=x.dtype, device=x.device)
    return torch.exp(-nu + torch.xlogy(x - 1, nu) - torch.lgamma(x))  # xlogy: 0 · log 0 = 0 at ν = 0


# --- both: the (S, L) state ---


def rhs(P: torch.Tensor, idx: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """d/ds of the (2, L) state: row 0 step growth (s = τ), row 1 living growth (s = ν)."""
    return torch.stack([step_growth_rhs(P[0], idx, mask), living_rhs(P[1])])


def moments(P: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    r"""μ_k = Σ_x x^k P_x for k = 0, 1, 2 (x summed):  einsum('...x,kx->...k').   (..., L) → (..., 3)."""
    powers = torch.stack([torch.ones_like(x), x, x**2])  # (3, L)
    return torch.einsum("...x,kx->...k", P, powers)


def averages(P: torch.Tensor, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """X_n = μ1/μ0, X_w = μ2/μ1 and the dispersity Đ = X_w / X_n."""
    m = moments(P, x)
    xn, xw = m[..., 1] / m[..., 0], m[..., 2] / m[..., 1]
    return xn, xw, xw / xn


def weight_fraction(P: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """w_x = x P_x / Σ_y y P_y."""
    xP = x * P
    return xP / xP.sum(-1, keepdim=True)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/chemistry/polymerisation_mwd/tensors.py
"""Build the concrete tensors: chain lengths, the shift (Toeplitz) index and the initial populations.

The state is one (S, L) tensor: row s = 0 holds step-growth chains, row s = 1 living chain-growth chains, and
column x - 1 the number of chains of length x (x = 1 … L monomer units), per initial monomer (step growth) or per
initiator (living). Both rows start as unimers: P_x(0) = δ_{x,1}.
"""

import torch

STEP, LIVING = 0, 1


def lengths(L: int, device: torch.device) -> torch.Tensor:
    """x = 1, 2, …, L as float64, shape (L,)."""
    return torch.arange(1, L + 1, dtype=torch.float64, device=device)


def shift_index(L: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Index tensor for T_{xy} = P_{x-y}: idx (L, L) and a mask that is 1 only where 1 ≤ x - y ≤ L.

    With columns 0 … L-1 standing for lengths 1 … L, a chain of length x+1 forms from lengths y+1 and x-y,
    i.e. column x - y - 1 (needs x - y ≥ 1).
    """
    x = torch.arange(L, device=device)
    diff = x[:, None] - x[None, :] - 1  # column of the partner chain
    mask = (diff >= 0).to(torch.float64)
    return diff.clamp_min(0), mask


def initial_state(L: int, device: torch.device) -> torch.Tensor:
    """P_{s,x}(0) = δ_{x,1} for both mechanisms: shape (2, L)."""
    P = torch.zeros(2, L, dtype=torch.float64, device=device)
    P[:, 0] = 1.0
    return P


def chain_length_cap(p_max: float) -> int:
    """Enough columns L that the Flory tail p^L at the final conversion is negligible (≲ 1e-7 of the chains)."""
    xn = 1.0 / (1.0 - p_max)
    return int(min(2000, max(200, 16 * xn + 100)))


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/chemistry/polymerisation_mwd/solve.py
"""Integrate the population balance with RK4, and a kinetic Monte Carlo of step growth for comparison."""

import torch

from problems._common.integrators import integrate
from problems._common.progress import steps

from .equation import rhs


def population_balance(P0: torch.Tensor, idx: torch.Tensor, mask: torch.Tensor, s_max: float, dt: float):
    """Return (s (T,), P (T, 2, L)): both mechanisms advanced together from s = 0 to s_max = X_n,max - 1."""
    n_steps = max(1, int(round(s_max / dt)))
    dt = s_max / n_steps
    traj = integrate(lambda P, s: rhs(P, idx, mask), P0, dt, n_steps)
    s = torch.arange(n_steps + 1, dtype=P0.dtype, device=P0.device) * dt
    return s, traj


def kinetic_monte_carlo(n_monomers: int, events_at: list[int], L: int, seed: int) -> torch.Tensor:
    """Step growth one bond at a time: pick an ordered pair of distinct chains (the A end of one, the B end of
    the other) uniformly at random and join them. With equal reactivity every pair is equally likely, so this
    is an exact stochastic simulation of the population balance (time is not needed: after e events p = e/N_0).

    Returns chain counts per initial monomer, shape (len(events_at), L), at the requested numbers of events.
    """
    gen = torch.Generator().manual_seed(seed)
    total = max(events_at)
    u = torch.rand(total, 2, generator=gen, dtype=torch.float64).tolist()  # random numbers drawn up front
    chains = [1] * n_monomers  # chain lengths; order does not matter
    wanted = sorted(set(events_at))
    snaps = {}
    if 0 in wanted:
        snaps[0] = list(chains)
    for e in steps(total, "KMC bonds"):
        N = len(chains)
        i = int(u[e][0] * N)
        j = int(u[e][1] * (N - 1))
        j += j >= i  # a different chain
        chains[i] += chains[j]  # join j onto i ...
        chains[j] = chains[-1]  # ... and fill j's slot with the last chain
        chains.pop()
        if e + 1 in wanted:
            snaps[e + 1] = list(chains)
    counts = []
    for e in events_at:
        lengths = torch.tensor(snaps[e]).clamp_max(L + 1)  # anything longer than L lands in an overflow bin
        counts.append(torch.bincount(lengths, minlength=L + 2)[1 : L + 1].to(torch.float64) / n_monomers)
    return torch.stack(counts)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/chemistry/polymerisation_mwd/plot.py
"""Animate polymerisation: the step-growth and living distributions on a common x / X_n axis, Đ and Carothers."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import frame_indices, save_gif

BLUE, RED, ORANGE, GREY, DARK = "#2563eb", "#dc2626", "#ea580c", "#9ca3af", "#374151"
U_MAX = 4.0  # right edge of the x / X_n axis


def animate(x, p_k, s_k, P_k, poisson, kmc, p_kmc, xn, disp, xn_mc, disp_mc, path: Path) -> Path:
    xs = x.cpu().numpy()  # (L,)
    p, s = p_k.cpu().numpy(), s_k.cpu().numpy()  # (K,)
    P, mc, po = P_k.cpu().numpy(), kmc.cpu().numpy(), poisson.cpu().numpy()  # (K, 2, L), (K, L), (K, L)
    pm = p_kmc.cpu().numpy()
    Xn, D = xn.cpu().numpy(), disp.cpu().numpy()  # (K, 2)
    Xn_mc, D_mc = xn_mc.cpu().numpy(), disp_mc.cpu().numpy()  # (K,)
    xn_end = Xn[-1, 0]

    def density(Pk: np.ndarray, xn_k: float) -> np.ndarray:
        """X_n w(x): the weight distribution as a density in u = x / X_n (area 1)."""
        w = xs * Pk / max((xs * Pk).sum(), 1e-300)
        return xn_k * w

    fig = plt.figure(figsize=(8.4, 4.6))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.2, 1])
    ax_s = fig.add_subplot(grid[0, 0])
    ax_l = fig.add_subplot(grid[1, 0])
    ax_d = fig.add_subplot(grid[0, 1])
    ax_c = fig.add_subplot(grid[1, 1])

    # --- step growth: ODE, KMC, Flory ---
    (ode_s,) = ax_s.plot([], [], color=BLUE, lw=1.8, label="population balance")
    (mc_s,) = ax_s.plot([], [], "o", color=ORANGE, ms=2.5, label="kinetic Monte Carlo")
    (fl_s,) = ax_s.plot([], [], "--", color=DARK, lw=1, label="Flory $x(1-p)^2p^{x-1}$")
    ax_s.set_xlim(0, U_MAX)
    ax_s.set_ylim(0, 0.6)
    ax_s.set_title("step growth (A–B monomers)", fontsize=9)
    ax_s.set_ylabel("$X_n\\,w(x)$", fontsize=8)
    ax_s.tick_params(labelsize=7)
    ax_s.legend(fontsize=6.5, loc="upper right")
    text_s = ax_s.text(0.02, 0.88, "", transform=ax_s.transAxes, fontsize=8)

    # --- living chain growth: ODE, Poisson ---
    (ode_l,) = ax_l.plot([], [], color=RED, lw=1.8, label="population balance")
    (po_l,) = ax_l.plot([], [], "--", color=DARK, lw=1, label="Poisson")
    (fl_l,) = ax_l.plot([], [], color=BLUE, lw=1, alpha=0.35, label="step growth, same $X_n$")
    ax_l.set_xlim(0, U_MAX)
    ax_l.set_title("living chain growth (no termination)", fontsize=9)
    ax_l.set_xlabel("chain length relative to the mean, $x / X_n$", fontsize=8)
    ax_l.set_ylabel("$X_n\\,w(x)$", fontsize=8)
    ax_l.tick_params(labelsize=7)
    ax_l.legend(fontsize=6.5, loc="upper right")
    text_l = ax_l.text(0.02, 0.88, "", transform=ax_l.transAxes, fontsize=8)

    # --- dispersity against X_n ---
    xg = np.linspace(1, xn_end, 300)
    nu = xg - 1
    ax_d.plot(xg, 2 - 1 / xg, "--", color=BLUE, lw=1, alpha=0.6)
    ax_d.plot(xg, 1 + nu / (1 + nu) ** 2, "--", color=RED, lw=1, alpha=0.6)
    ax_d.plot(xg, 1 + 1 / xg, ":", color=RED, lw=1, alpha=0.6)
    ax_d.text(xn_end, 2.03, "Đ = 1 + p → 2", ha="right", va="bottom", fontsize=7, color=BLUE)
    ax_d.text(xn_end, 1.04, "Đ ≈ 1 + 1/$X_n$", ha="right", va="bottom", fontsize=7, color=RED)
    (d_step,) = ax_d.plot([], [], color=BLUE, lw=1.8)
    (d_liv,) = ax_d.plot([], [], color=RED, lw=1.8)
    (d_mc,) = ax_d.plot([], [], "o", color=ORANGE, ms=2.5)
    ax_d.set_xscale("log")
    ax_d.minorticks_off()
    ax_d.set_xlim(1, xn_end * 1.05)
    ax_d.set_ylim(0.95, 2.15)
    ax_d.set_title("dispersity Đ = $X_w / X_n$", fontsize=9)
    ax_d.tick_params(labelsize=7)

    # --- Carothers ---
    pg = np.linspace(0, p[-1], 300)
    ax_c.plot(pg, 1 / (1 - pg), color=GREY, lw=2.5, alpha=0.6, label="Carothers $1/(1-p)$")
    (c_ode,) = ax_c.plot([], [], "o", color=BLUE, ms=5, label="population balance")
    (c_mc,) = ax_c.plot([], [], "x", color=ORANGE, ms=6, label="KMC")
    ax_c.set_xlim(0, 1)
    ax_c.set_ylim(0, xn_end * 1.1)
    ax_c.set_xlabel("conversion p", fontsize=8)
    ax_c.set_ylabel("$X_n$", fontsize=8)
    ax_c.tick_params(labelsize=7)
    ax_c.legend(fontsize=6.5, loc="upper left")

    bins = np.unique(np.round(np.geomspace(1, xs[-1] + 1, 70)).astype(int))  # lumps for the noisy KMC tail

    def update(k: int):
        xn_k = Xn[k, 0]
        u = xs / xn_k
        ode_s.set_data(u, density(P[k, 0], xn_k))
        fl_s.set_data(u, xn_k * xs * (1 - p[k]) ** 2 * p[k] ** (xs - 1))
        w_mc = xs * mc[k] / max((xs * mc[k]).sum(), 1e-300)  # KMC weight fraction, then averaged over bins
        centres, dens = [], []
        for lo, hi in zip(bins[:-1], bins[1:]):
            if hi > lo:
                centres.append(0.5 * (lo + hi - 1))
                dens.append(w_mc[lo - 1 : hi - 1].sum() / (hi - lo))
        mc_s.set_data(np.array(centres) / Xn_mc[k], np.array(dens) * Xn_mc[k])
        text_s.set_text(f"p = {p[k]:.3f}   $X_n$ = {xn_k:.1f}   Đ = {D[k, 0]:.3f}")

        dl = density(P[k, 1], Xn[k, 1])
        ode_l.set_data(xs / Xn[k, 1], dl)
        po_l.set_data(xs / Xn[k, 1], density(po[k], Xn[k, 1]))
        fl_l.set_data(u, density(P[k, 0], xn_k))
        ax_l.set_ylim(0, max(0.6, 1.15 * dl.max()))
        text_l.set_text(f"ν = {s[k]:.1f}   $X_n$ = {Xn[k, 1]:.1f}   Đ = {D[k, 1]:.3f}")

        d_step.set_data(Xn[: k + 1, 0], D[: k + 1, 0])
        d_liv.set_data(Xn[: k + 1, 1], D[: k + 1, 1])
        d_mc.set_data(Xn_mc[: k + 1], D_mc[: k + 1])
        c_ode.set_data([p[k]], [xn_k])
        c_mc.set_data([pm[k]], [Xn_mc[k]])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(p), 90), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'p_max': 0.98,
    'n_monomers': 100000
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.chemistry.polymerisation_mwd")]:
    del sys.modules[name]
main = importlib.import_module("problems.chemistry.polymerisation_mwd.main")
problem_dir = Path("problems/chemistry/polymerisation_mwd")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
